<br>
<p float="right">
  <img src=attachment:nanoHUB_logo_color.png width="25%" height='10%' align="right" /> 
</p>

# Demonstration: LAMMPS Parser on LLM-Automatic Script Generation

### <i>Ethan Holbrook, Juan C. Verduzco, </i>  and <i>Alejandro Strachan </i>
### Materials Engineering, Purdue University <br>

## Overview

This notebook is the fourth stage of the evaluation pipeline. It evaluates whether successfully parsed and executed scripts match the intended simulation setup described in each benchmark prompt.

Accuracy is assessed with prompt-specific rules derived from the generated ASTs, including checks on quantities such as lattice type, lattice parameter, ensemble controls, runtime settings, and other required simulation details. The outputs of this notebook form the basis for the final benchmark labels used in downstream summaries and visualizations.

Accuracy functions are primarily intended to determine which scripts do not have any errors. They are not meant to find all errors within a specific script and have many hard-coded values that may make broad error statistics difficult to obtain. 

## Tips
1. Found a bug? Email holbrooe@purdue.edu

<br><br><br>

# Libraries

In [1]:
import os
import json
import sys
import shutil
import re

from dotenv import load_dotenv
load_dotenv()

from lark import tree
from lammps_ast.sanitizer import sanitize
from lammps_ast.parser import parse_to_AST

import lammps_ast
print(dir(lammps_ast))
print(lammps_ast.__path__)

import openai
from openai import OpenAI

import numpy as np
import pandas as pd

import subprocess

from colorama import Fore, Style

from importlib.metadata import version
print(version("lammps-ast"))

# import anthropic

import ast # python ast

['__author__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'error_handler', 'grammar', 'parse_to_AST', 'parser', 'sanitize', 'sanitizer', 'transformer']
['/home/holbrooe/.conda/envs/2022.10-py39/gst/lib/python3.12/site-packages/lammps_ast']
0.1.7


In [2]:
# read execution results
pair_df = pd.read_pickle('final_pair_df.pkl')

## Initialization of directory variables

# Errors

In [7]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir))
scripts_dir = os.path.join(parent_dir,"generated_scripts")
scripts_base = os.path.join(parent_dir,"generated_scripts")

print(scripts_dir)

prompt_dirs = ['prompt1','prompt2','prompt3']
model_dirs = ['gpt-4o','gpt-4.1','gpt-o3','claude-opus-4','gpt-5'] #,'gpt-4o-search',]
# model_dirs = [] #,'gpt-4o-search',]
model_names = ['gpt-4o-2024-08-06','gpt-4.1-2025-04-14','o3-2025-04-16','claude-4-opus-20250514','gpt-5-2025-08-07']
# model_names = ['gpt-5-2025-08-07']

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts


In [4]:
prompt_choice = 1
model_choice = 0

prompt_type = prompt_dirs[prompt_choice-1]
print(prompt_type)
model_type = model_dirs[model_choice]
model_name = model_names[model_choice]
print(model_type,model_name)

trial=0
# output_filename = os.path.join(prompt_type, model_type,f'P_{prompt_type}-M_{model_type}-T_{trial}.in')

print(prompt_dirs)
print(model_dirs)
print(model_names)

# prompt_dict = {'prompt1':prompts[0],'prompt2':prompts[1],'prompt3':prompts[2]}

model_dict = {model_dirs[0]:model_names[0],
              model_dirs[1]:model_names[1],
              model_dirs[2]:model_names[2],
              model_dirs[3]:model_names[3],
              model_dirs[4]:model_names[4],
             }

prompt1
gpt-4o gpt-4o-2024-08-06
['prompt1', 'prompt2', 'prompt3']
['gpt-4o', 'gpt-4.1', 'gpt-o3', 'claude-opus-4', 'gpt-5']
['gpt-4o-2024-08-06', 'gpt-4.1-2025-04-14', 'o3-2025-04-16', 'claude-4-opus-20250514', 'gpt-5-2025-08-07']


In [8]:
#what we see
print(scripts_dir)
prompt_numbers = sorted(next(os.walk(scripts_dir))[1])
print(prompt_numbers)

prompt_model_map1 = {}

for prompt in prompt_numbers:
    prompt_dir = os.path.join(scripts_dir, prompt)
    models = sorted(next(os.walk(prompt_dir))[1])  
    prompt_model_map1[prompt] = models

# Display detected structure
for prompt, models in prompt_model_map1.items():
    print(f"📂 {prompt}: {', '.join(models)}")

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts
['prompt1', 'prompt2', 'prompt3']
📂 prompt1: gpt-4o
📂 prompt2: gpt-4o
📂 prompt3: gpt-4o


In [9]:
accuracy_df = pair_df.copy() # Comment if loading from pickle, if not must use
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(accuracy_df)

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True,n/a
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...,[Errno 2] No such file or directory: '/apps/sh...
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...,[Errno 2] No such file or directory: '/apps/sh...


In [10]:
import os, pickle
from lark import Tree, Token
import ast

def _tokens_under(node):
    return [x for x in node.scan_values(lambda v: isinstance(v, Token))]

def _iter_statement_nodes(ast_tree):
    for t in ast_tree.iter_subtrees():
        if isinstance(t, Tree) and str(t.data).endswith("_statement"):
            toks = _tokens_under(t)
            parts = [tok.value for tok in toks if tok.value is not None]
            if parts:
                yield str(t.data), parts

def _cmd_map_from_ast(ast_tree):
    cmd_map = {}
    for stmt_name, parts in _iter_statement_nodes(ast_tree):
        cmd = parts[0].lower()
        cmd_map.setdefault(cmd, []).append(parts)
    return cmd_map


def accuracy_from_ast_row(
    row,
    expected_fix=None, fix_tolerances=None,
    expected_lattice=4.05, lattice_tol=.1,
    expected_timestep=0.001, timestep_tol=1e-6,
    expected_run=500000, run_tol=0,
    expected_total_time=None, total_time_tol=1e-4,
    expected_pair_style="eam/alloy",
):
    """
    Validate key LAMMPS settings by reading the stored AST for this row.
    Returns:
      - "yes" if everything matches expectations
      - comma-separated issue codes otherwise
      - "not parsed or not executed" if the row is ineligible or if the AST file is missing
    """

    issues = []
    
    # --- Elegibility -----
#     if row.get("run") != "Success" and row.get("pair_run") == "Success":
    if row.get("run") != True and row.get("pair_run") == True:
        issues.append("PSZ")  
    # --- (0) Bonus Thermostyle out-of-order
    if row.get("run") != True and row.get("pair_run") != True and row.get("run") != 'not parsed':
        error_list = ast.literal_eval(row.get("run"))
        if error_list[0].startswith('ERROR: Thermo_style command before'):
            issues.append('bad thermo order')
        elif error_list[0].startswith('ERROR on proc 0: Neighbor list overflow'):
            issues.append('lost atoms')
        elif error_list[0].startswith('ERROR: Incorrect args for pair coefficients'):
            pass # covered with parser
        else:
            issues.append('unknown error')

    ast_path = row.get("ast_path")
    if not ast_path or not os.path.exists(ast_path):
        return "no AST generated"

    # --- Defaults ---
    if expected_fix is None:
        expected_fix = dict( temp_start=300.0, temp_end=300.0, temp_damp=0.1, pressure_start=1.0, pressure_end=1.0, pressure_damp=1.0)
    
    if fix_tolerances is None:
        fix_tolerances = dict( temp_start=1e-4, temp_end=1e-4, temp_damp=1e-4, pressure_start=.2, pressure_end=.2, pressure_damp=1e-4)
        
    if expected_total_time is None:
        expected_total_time = expected_timestep * expected_run

    # --- Load AST / Command map ---
    with open(ast_path, "rb") as f:
        LAMMPS_AST = pickle.load(f)

    cmd_map = _cmd_map_from_ast(LAMMPS_AST)
    

    # --- (1) Check timestep ---
    timestep_val = expected_timestep
    
    if "timestep" in cmd_map:
        try:
            timestep_val = float(cmd_map["timestep"][0][1])
            if abs(timestep_val - expected_timestep) > timestep_tol:
                issues.append("timestep inaccurate")
        except Exception:
            issues.append("timestep parsing error")         
            
    # --- (2) Check run ---
    run_val = None
    
    if "run" in cmd_map:
        try:
            run_val = int(cmd_map["run"][0][1])
            if abs(run_val - expected_run) > run_tol:
                issues.append("run inaccurate")
        except Exception as e:
            print(e)
            issues.append("run parsing error")
    else:
        issues.append("run missing")

    # --- (3) Check pair_style ---
    if "pair_style" in cmd_map:
        ps = cmd_map["pair_style"][0]
        if len(ps) < 2 or ps[1] != expected_pair_style:
            issues.append("pair_style inaccurate")
    else:
        issues.append("pair_style missing")

    # --- (4) Check lattice ---
    if "lattice" in cmd_map:
        lat = cmd_map["lattice"][0]

        if len(lat) < 3 or lat[1].lower() != "fcc":
            issues.append("lattice type inaccurate")

        try:
            lat_val = float(lat[2])
            if abs(lat_val - expected_lattice) > lattice_tol:
                issues.append("lattice_param inaccurate")
        except Exception:
            issues.append("lattice_param parsing error")
    else:
        issues.append("lattice missing")

    # --- (5) Check fix npt temp/iso values ---
    
    npt_line = None
    for parts in cmd_map.get("fix", []):
        lower = [p.lower() for p in parts]
        if "npt" in lower and "temp" in lower: ######### NPT IS hardcoded - Ethan
            npt_line = parts
            break

    if npt_line is None:
        issues.append("fix missing")
    else:
        try:
            lower = [p.lower() for p in npt_line]
            i_temp = lower.index("temp")
            i_iso = lower.index("iso")

            parsed_fix = {
                "temp_start": float(npt_line[i_temp + 1]),
                "temp_end": float(npt_line[i_temp + 2]),
                "temp_damp": float(npt_line[i_temp + 3]),
                "pressure_start": float(npt_line[i_iso + 1]),
                "pressure_end": float(npt_line[i_iso + 2]),
                "pressure_damp": float(npt_line[i_iso + 3]),
            }

            for key in expected_fix:
                if abs(parsed_fix[key] - expected_fix[key]) > fix_tolerances[key]:
                    issues.append(key)

        except Exception:
            issues.append("fix_parse")
            
    # --- (6) Check size ---
    
    region_dims = None
    replicate_vals = None

    # region parsing (take first matching "region ... block ..." line)
    if "region" in cmd_map:
        for reg in cmd_map["region"]:
            # expect: ['region', <name>, 'block', xlo, xhi, ylo, yhi, zlo, zhi, ...]
            try:
                if len(reg) >= 9 and reg[2].lower() == "block":
                    xhi = float(reg[4])
                    yhi = float(reg[6])
                    zhi = float(reg[8])
                    region_dims = (xhi, yhi, zhi)
                    break
            except Exception:
                issues.append("size_parse")

    if "replicate" in cmd_map:
        rep = cmd_map["replicate"][0]
        try:
            if len(rep) >= 4:
                replicate_vals = (int(rep[1]), int(rep[2]), int(rep[3]))
        except Exception:
            issues.append("replicate_parse")

    # apply your size logic
    if region_dims is not None:
        if region_dims == (1.0, 1.0, 1.0):
            if replicate_vals != (5, 5, 5):
                issues.append("size")
        elif region_dims == (5.0, 5.0, 5.0):
            if replicate_vals not in [None, (1, 1, 1)]:
                issues.append("size")
        else:
            issues.append("size")
    else:
        issues.append("size_missing")

    # --- (6) Check total sim time ---

    if timestep_val is not None and run_val is not None:
        total_time = timestep_val * run_val
        if abs(total_time - expected_total_time) > total_time_tol:
            issues.append("total_time")
    else:
        issues.append("total_time_incomplete")   
        


    return True if not issues else ", ".join(sorted(set(issues)))


In [12]:
def add_accuracy_from_ast(df, **kwargs):
    df = df.copy()
    df["accurate"] = df.apply(lambda r: accuracy_from_ast_row(r, **kwargs), axis=1)
    return df

def highlight_unparsed(row):
    if row["parsed"] != True:
        return ["background-color: #ffe6e6"] * len(row)  # light red
    return [""] * len(row)

accuracy_add_df = add_accuracy_from_ast(accuracy_df[accuracy_df['prompt'] == 'prompt1'])

a = accuracy_add_df.style.apply(highlight_unparsed, axis=1)
display(a)

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True,n/a,True


In [13]:
import math 

def accuracy_from_ast_row_prompt2(
    row,
    expected_fix=None, fix_tolerances=None,
    expected_lattice=3.52, lattice_tol=.1,
    expected_timestep=0.001, timestep_tol=1e-6,
    expected_run=220000, run_tol=0,
    expected_total_time=None, total_time_tol=1e-4,
    expected_pair_style="eam/alloy",
    expected_velocity_temp=600.0, velocity_tol=1e-6,
    rate_K_per_ps=10.0,rate_tol=0.25,
):
    """
    Validate key LAMMPS settings by reading the stored AST for this row.
    Returns:
      - "True" if everything matches expectations
      - comma-separated issue codes otherwise
      - "not parsed or not executed" if the row is ineligible or if the AST file is missing
    """

    issues = []
    
    # --- Elegibility -----
    if row.get("run") != True and row.get("pair_run") == True:
        issues.append("PSZ")    
    # --- (0) Bonus Thermostyle out-of-order --and other execution errors
    if row.get("run") != True and row.get("pair_run") != True and row.get("run") != 'not parsed':
        error_list = ast.literal_eval(row.get("run"))
        if error_list[0].startswith('ERROR: Thermo_style command before'):
            issues.append('bad thermo order')
        elif error_list[0].startswith('ERROR on proc 0: Neighbor list overflow'):
            issues.append('lost atoms')
        elif error_list[0].startswith('ERROR: Incorrect args for pair coefficients'):
            pass # covered with parser - element listed
        elif error_list[0].startswith('ERROR on proc 0: Not a valid integer number:'):
            pass # covered with parser - no element listed
        else:
            issues.append('unknown error')
            
    ast_path = row.get("ast_path")
    if not ast_path or not os.path.exists(ast_path):
        return "no AST generated"

    # --- Defaults ---
    if expected_fix is None:
        expected_fix = dict( temp_start=300.0, temp_end=2500.0, temp_damp=0.1, pressure_start=1.0, pressure_end=1.0, pressure_damp=1.0)
    
    if fix_tolerances is None:
        fix_tolerances = dict( temp_start=1e-4, temp_end=1e-4, temp_damp=1, pressure_start=.2, pressure_end=.2, pressure_damp=10)
        
    if expected_total_time is None:
        expected_total_time = expected_timestep * expected_run

    lattice_type = 'fcc' #P1/P2 BCC P3
    # --- Load AST / Command map ---
    with open(ast_path, "rb") as f:
        LAMMPS_AST = pickle.load(f)

    cmd_map = _cmd_map_from_ast(LAMMPS_AST)
    
    # --- (0.5) Units ---
    if "units" in cmd_map:
        units_val = cmd_map["units"][0][1]
        if units_val == 'metal':
            pass
        else:
            issues.append('units')
    
    # --- (1) Check timestep ---
    timestep_val = expected_timestep
    
    if "timestep" in cmd_map:
        try:
            timestep_val = float(cmd_map["timestep"][0][1])
            if abs(timestep_val - expected_timestep) > timestep_tol:
                issues.append("timestep inaccurate")
        except Exception:
            issues.append("timestep parsing error")         
           
    # --- (2) Check run ---
    run_val = None
    
    if "run" in cmd_map:
        try:
            run_val = int(cmd_map["run"][0][1])
            if abs(run_val - expected_run) > run_tol:
                issues.append("run num inaccurate")
        except Exception:
            issues.append("run parsing error")
    else:
        issues.append("run missing")

    # --- (3) Check pair_style ---
    if "pair_style" in cmd_map:
        ps = cmd_map["pair_style"][0]
        if len(ps) < 2 or ps[1] != expected_pair_style:
            issues.append("pair_style inaccurate")
    else:
        issues.append("pair_style missing")

    # --- (4) Check lattice ---
    if "lattice" in cmd_map:
        lat = cmd_map["lattice"][0]

        if len(lat) < 3 or lat[1].lower() != lattice_type:
            issues.append("lattice type inaccurate")

        try:
            lat_val = float(lat[2])
            if abs(lat_val - expected_lattice) > lattice_tol:
                issues.append("lattice_param inaccurate")
        except Exception:
            issues.append("lattice_param parsing error")
    else:
        issues.append("lattice missing")

    # --- (5) Check fix npt temp/iso values ---
    
    npt_line = None
    for parts in cmd_map.get("fix", []):
        lower = [p.lower() for p in parts]
        if "npt" in lower and "temp" in lower: ######### NPT IS hardcoded - Ethan -- What part?
            npt_line = parts # I really think it handles the ID fine
#             print(npt_line) # get to see the different ID names
            break

    if npt_line is None:
        issues.append("fix missing")
        print('fix_missing')
    else:
        try:
            lower = [p.lower() for p in npt_line]
            i_temp = lower.index("temp")
            i_iso = lower.index("iso")

            parsed_fix = {
                "temp_start": float(npt_line[i_temp + 1]),
                "temp_end": float(npt_line[i_temp + 2]),
                "temp_damp": float(npt_line[i_temp + 3]),
                "pressure_start": float(npt_line[i_iso + 1]),
                "pressure_end": float(npt_line[i_iso + 2]),
                "pressure_damp": float(npt_line[i_iso + 3]),
            }

            for key in expected_fix:
                if abs(parsed_fix[key] - expected_fix[key]) > fix_tolerances[key]:
                    issues.append(key)
        
        except Exception:
            issues.append("fix_parse")
            
    # --- (5.5) Velocity assignment check
    vel_line = None
    for parts in cmd_map.get("velocity", []):
        lower = [p.lower() for p in parts]
        # match: velocity all create <T> <seed> ...
        if len(lower) >= 5 and lower[1] == "all" and lower[2] == "create":
            vel_line = parts
            break

    if vel_line is None:
        issues.append("velocity missing")
    else:
        try:
            vT = float(vel_line[3])
            if abs(vT - expected_velocity_temp) > velocity_tol:
                issues.append("velocity_temp inaccurate")
        except Exception:
            issues.append("velocity parsing error")
    
    # --- (6) Check size ---
    
    region_dims = None
    replicate_vals = None

    # region parsing (take first matching "region ... block ..." line)
    if "region" in cmd_map:
        for reg in cmd_map["region"]:
            # expect: ['region', <name>, 'block', xlo, xhi, ylo, yhi, zlo, zhi, ...]
            try:
                if len(reg) >= 9 and reg[2].lower() == "block":
                    xhi = float(reg[4])
                    yhi = float(reg[6])
                    zhi = float(reg[8])
                    region_dims = (xhi, yhi, zhi)
                    break
            except Exception:
                issues.append("size_parse")

    if "replicate" in cmd_map:
        rep = cmd_map["replicate"][0]
        try:
            if len(rep) >= 4:
                replicate_vals = (int(rep[1]), int(rep[2]), int(rep[3]))
        except Exception:
            issues.append("replicate_parse")

    # apply your size logic
    if region_dims is not None: # nicle unit cell 10 times in each direction
        if region_dims == (1.0, 1.0, 1.0):
            if replicate_vals != (10, 10, 10):
                issues.append("size")
        elif region_dims == (10.0, 10.0, 10.0):
            if replicate_vals not in [None, (1, 1, 1)]:
                issues.append("size")
        else:
            issues.append("size")
    else:
        issues.append("size_missing")

    # --- (6) Check total sim time & Heating Rate ---

    if timestep_val is not None and run_val is not None:
        total_time = timestep_val * run_val
        if abs(total_time - expected_total_time) > total_time_tol:
            issues.append("sim time inaccurate")
        if parsed_fix:
            dT = parsed_fix["temp_end"] - parsed_fix["temp_start"]
#             print(dT)
            rate = dT / total_time if total_time > 0 else float('inf')
#             print(rate)
            if not math.isclose(rate, rate_K_per_ps, rel_tol=0, abs_tol=rate_tol):
                issues.append('heating_rate')
#                 print('hr')
    else:
        issues.append("total_time_incomplete") 
    
#     print(issues)
    
    return True if not issues else ", ".join(sorted(set(issues)))


In [14]:
def add_accuracy_from_ast_prompt2(df, **kwargs):
    df = df.copy()
    df["accurate"] = df.apply(lambda r: accuracy_from_ast_row_prompt2(r, **kwargs), axis=1)
    return df

def highlight_unparsed(row):
    if row["parsed"] != True:
        return ["background-color: #ffe6e6"] * len(row)  # light red
    return [""] * len(row)

accuracy_add2_df = add_accuracy_from_ast_prompt2(accuracy_df[accuracy_df['prompt'] == 'prompt2'])

a = accuracy_add2_df.style.apply(highlight_unparsed, axis=1)
display(a)

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,"['ERROR: Incorrect args for pair coefficients (src/src/MANYBODY/pair_eam.cpp:374)', 'Last command: pair_coeff * * potentials/prompt2.potential Ni']",[Errno 2] No such file or directory: '/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial',"pair_style inaccurate, size"


Checklist:
- Nickel
- Lattice Parameter ~ 3.52 Å
- size: 10 Ni unitcells
- Lattice Type: FCC
- Velocity: Maxwell-Boltzmann distribution at 600 K
- NPT from 300K to 2500K at 1 atm
- Rate: 10K per ps
- EAM/ALLOY potential 

In [15]:
import math 

def accuracy_from_ast_row_prompt3(
    row,
    expected_lattice=3.3, lattice_tol=.1,
    expected_timestep=0.001, timestep_tol=1e-6,
    expected_run1=100000, run_tol1=0,
    expected_total_time1=100, total_time_tol1=1e-4, # in ps
    
    expected_run2=None, run_tol2=0,
    expected_total_time2=None, total_time_tol2=1e-4,
    
    expected_pair_style="eam/alloy",
    
    expected_velocity_temp=300, velocity_tol=1e-6,
    lattice_type = 'bcc', #P1/P2 BCC P3
    velocity_add_magnitude=2, # km/s
):
    """
    Validate key LAMMPS settings by reading the stored AST for this row.
    Returns:
      - "True" if everything matches expectations
      - comma-separated issue codes otherwise
      - "not parsed or not executed" if the row is ineligible or if the AST file is missing
    """
    KM_S_TO_A_PS = 10.0 # 1 km/s = 10 Å/ps
    expected_rel_speed_A_ps = velocity_add_magnitude * KM_S_TO_A_PS
    
    issues = []
    
    # --- Elegibility -----
    if row.get("run") != True and row.get("pair_run") == True:
        issues.append("PSZ")    
    # --- (0) Bonus Thermostyle out-of-order --and other execution errors
    if row.get("run") != True and row.get("pair_run") != True and row.get("run") != 'not parsed':
        error_list = ast.literal_eval(row.get("run"))
        if error_list[0].startswith('ERROR: Thermo_style command before'):
            issues.append('bad thermo order')
        elif error_list[0].startswith('ERROR on proc 0: Neighbor list overflow'):
            issues.append('lost atoms')
        elif error_list[0].startswith('ERROR: Incorrect args for pair coefficients'):
            pass # covered with parser - element listed
        elif error_list[0].startswith('ERROR on proc 0: Not a valid integer number:'):
            pass # covered with parser - no element listed
        else:
            issues.append('unknown error')
            
    ast_path = row.get("ast_path")
    if not ast_path or not os.path.exists(ast_path):
        return "no AST generated"

    # --- Fix 1 100ps NVT @ 300K ---

    expected_fix = dict( temp_start=300.0, temp_end=300.0, temp_damp=0.1 )
        
    fix_tolerances = dict( temp_start=1e-4, temp_end=1e-4, temp_damp=1 )
        

    # --- Fix 2 NVE for some time ---

    # nothing
        
    # --- Load AST / Command map ---
    with open(ast_path, "rb") as f:
        LAMMPS_AST = pickle.load(f)

    cmd_map = _cmd_map_from_ast(LAMMPS_AST)
    
    # --- (0.5) Units ---
    if "units" in cmd_map:
        units_val = cmd_map["units"][0][1]
        if units_val == 'metal':
            pass
        else:
            issues.append('units')
    
    # --- (1) Check timestep ---
    timestep_val = expected_timestep
    
    if "timestep" in cmd_map:
        try:
            timestep_val = float(cmd_map["timestep"][0][1])
            if abs(timestep_val - expected_timestep) > timestep_tol:
                issues.append("timestep inaccurate")
        except Exception:
            issues.append("timestep parsing error")         
           
    # --- (2) Check NVT run ---
    run_val = None
    
    if "run" in cmd_map:
        try:
            run_val = int(cmd_map["run"][0][1])
            if abs(run_val - expected_run1) > run_tol1:
                issues.append("run num inaccurate")
        except Exception as e:
            print(e)
            issues.append("run parsing error")
            
#         print(len(cmd_map["run"]))
#         print(cmd_map["run"])
        if len(cmd_map["run"]) == 2:
            pass
        else:
            issues.append('not 2 runs') # should be one run for NVT and one for NVE
            # flags more or less
    else:
        issues.append("run missing")
        
        

    # --- (3) Check pair_style ---
    if "pair_style" in cmd_map:
        ps = cmd_map["pair_style"][0]
        if len(ps) < 2 or ps[1] != expected_pair_style:
            issues.append("pair_style inaccurate")
    else:
        issues.append("pair_style missing")

    # --- (4) Check lattice ---
    if "lattice" in cmd_map:
        lat = cmd_map["lattice"][0]

        if len(lat) < 3 or lat[1].lower() != lattice_type:
            issues.append("lattice type inaccurate")

        try:
            lat_val = float(lat[2])
            if abs(lat_val - expected_lattice) > lattice_tol:
                issues.append("lattice_param inaccurate")
        except Exception:
            issues.append("lattice_param parsing error")
    else:
        issues.append("lattice missing")

    # --- (5) Check fix npt temp/iso values ---
    
    nvt_line = None
    nve_line = None
    
    for parts in cmd_map.get("fix", []):
        lower = [p.lower() for p in parts]
#         print(lower)
        if "nvt" in lower: 
            nvt_line = parts # I really think it handles the ID fine
#             print(npt_line) # get to see the different ID names
        elif "nve" in lower:
            nve_line = parts
        else:
            print("Unaccounted fix: ",lower)

    ### NVT Fix
    if nvt_line is None:
        issues.append("nvt fix missing")
#         print('nvt fix missing')
    else:
        try:
            lower = [p.lower() for p in nvt_line]
            i_temp = lower.index("temp")

            parsed_fix = {
                "temp_start": float(nvt_line[i_temp + 1]),
                "temp_end": float(nvt_line[i_temp + 2]),
                "temp_damp": float(nvt_line[i_temp + 3]),
            }

            for key in expected_fix:
                if abs(parsed_fix[key] - expected_fix[key]) > fix_tolerances[key]:
                    issues.append(key)
        except Exception:
#             print(e)
            issues.append("fix_parse")
    ### NVE Fix
    if nve_line is None:
        issues.append("nve fix missing")
#         print('nve fix missing')
            
    # --- (5.5) Velocity assignment check
    vel_init_line = None
    vel_add_line = None
    vel_any_line = None
    for parts in cmd_map.get("velocity", []):
        lower = [p.lower() for p in parts]
        print(lower)
        # match: velocity all create <T> <seed> ...
        if len(lower) >= 5 and lower[1] == "all" and lower[2] == "create":
            vel_init_line = parts
#             break
        elif len(lower) >= 5 and lower[2] == "set":
            vel_add_line = parts
        else:
            vel_any_line = parts
            print(lower)
            issues.append('extra_vel')
        if 'add' in lower:
            issues.append('vel_add')
            
            
    if vel_init_line is None and vel_add_line is None and vel_any_line is None:
        issues.append("velocity missing")
    if vel_init_line is not None:
        try:
            vT = float(vel_init_line[3])
            if abs(vT - expected_velocity_temp) > velocity_tol:
                issues.append("velocity_temp inaccurate")
        except Exception as e:
            print(e)
            issues.append("velocity parsing error")
    if vel_add_line is not None:
        print(float(vel_add_line[5]))
        if abs(float(vel_add_line[5])) == expected_rel_speed_A_ps:
            pass
        else:
            issues.append('v_set_speed')
        if float(vel_add_line[5]) > 0:
            issues.append('v_set_direction')            
            
    # --- (6) Check size ---
    
#     region_dims = None
#     replicate_vals = None

#     # region parsing (take first matching "region ... block ..." line)
#     if "region" in cmd_map:
#         for reg in cmd_map["region"]:
#             # expect: ['region', <name>, 'block', xlo, xhi, ylo, yhi, zlo, zhi, ...]
#             try:
#                 if len(reg) >= 9 and reg[2].lower() == "block":
#                     xhi = float(reg[4])
#                     yhi = float(reg[6])
#                     zhi = float(reg[8])
#                     region_dims = (xhi, yhi, zhi)
#                     break
#             except Exception:
#                 issues.append("size_parse")

#     if "replicate" in cmd_map:
#         rep = cmd_map["replicate"][0]
#         try:
#             if len(rep) >= 4:
#                 replicate_vals = (int(rep[1]), int(rep[2]), int(rep[3]))
#         except Exception:
#             issues.append("replicate_parse")

#     # apply your size logic
#     if region_dims is not None: # nicle unit cell 10 times in each direction
#         if region_dims == (1.0, 1.0, 1.0):
#             if replicate_vals != (10, 10, 10):
#                 issues.append("size")
#         elif region_dims == (10.0, 10.0, 10.0):
#             if replicate_vals not in [None, (1, 1, 1)]:
#                 issues.append("size")
#         else:
#             issues.append("size")
#     else:
#         issues.append("size_missing")

#######################
    # --- (6) Check size (spall construction) ---
    #
    # Geometry expectations expressed in *lattice units*:
    #   - x: 0..20, y: 0..20
    #   - target z: 0..40  (40 cells)
    #   - gap: 15 Å -> gap_cells = 15 / a  (a = lattice parameter used in script)
    #   - projectile z: (40 + gap_cells)..(40 + gap_cells + 20)
    #   - box z: 0..(40 + gap_cells + 20)

    size_tol_lat = 1e-6  # tolerance in lattice units

    # pull lattice parameter actually used in the script (fallback to expected_lattice)
    lat_A = expected_lattice
    if "lattice" in cmd_map and len(cmd_map["lattice"]) >= 1:
        try:
            lat_A = float(cmd_map["lattice"][0][2])
        except Exception:
            issues.append("size_lattice_parse")

    proj_cells = 20.0
    targ_cells = 40.0
    gap_A = 15.0

    # avoid divide-by-zero
    if lat_A == 0.0:
        issues.append("size_lattice_zero")
        gap_cells = None
    else:
        gap_cells = gap_A / lat_A

    # parse regions by name
    box_bounds = None
    target_bounds = None
    projectile_bounds = None

    if "region" in cmd_map:
        for reg in cmd_map["region"]:
            try:
                if len(reg) >= 9 and reg[2].lower() == "block":
                    name = reg[1].lower()
                    bounds = (
                        float(reg[3]), float(reg[4]),
                        float(reg[5]), float(reg[6]),
                        float(reg[7]), float(reg[8]),
                    )
                    if name == "box":
                        box_bounds = bounds
                    elif name == "target":
                        target_bounds = bounds
                    elif name == "projectile":
                        projectile_bounds = bounds
            except Exception:
                issues.append("size_parse")

    # require the regions
    if box_bounds is None:
        issues.append("box_region_missing")
    if target_bounds is None:
        issues.append("target_region_missing")
    if projectile_bounds is None:
        issues.append("projectile_region_missing")

    # if we can compute gap_cells, validate the full layout
    if gap_cells is not None:
        total_z_cells = targ_cells + gap_cells + proj_cells

        exp_box = (0.0, 20.0, 0.0, 20.0, 0.0, total_z_cells)
        exp_target = (0.0, 20.0, 0.0, 20.0, 0.0, targ_cells)
        exp_proj = (0.0, 20.0, 0.0, 20.0, targ_cells + gap_cells, total_z_cells)

        if box_bounds is not None:
            for g, e in zip(box_bounds, exp_box):
                if abs(g - e) > size_tol_lat:
                    issues.append("box_region_size")
                    break

        if target_bounds is not None:
            for g, e in zip(target_bounds, exp_target):
                if abs(g - e) > size_tol_lat:
                    issues.append("target_region_size")
                    break

        if projectile_bounds is not None:
            for g, e in zip(projectile_bounds, exp_proj):
                if abs(g - e) > size_tol_lat:
                    issues.append("projectile_region_size")
                    break

        # gap consistency: projectile.zlo - target.zhi == gap_cells
        if target_bounds is not None and projectile_bounds is not None:
            gap_got = projectile_bounds[4] - target_bounds[5]
            if abs(gap_got - gap_cells) > size_tol_lat:
                issues.append("gap_size")
    else:
        issues.append("size_incomplete")

    # Optional: if you do not want replicate used in this construction, flag it.
    # (Comment out if you want to allow replicate-based builds.)
    if "replicate" in cmd_map:
        issues.append("replicate_unexpected")
####################

        
    if "boundary" in cmd_map:
        bound_cond = cmd_map['boundary'][0][1:4]
        if bound_cond == ['p','p','s']:
            pass
        elif bound_cond == ['p','p','f']:
            issues.append('fixed bounds')
        else:
            issues.append('other bounds')

    # --- (6) Check total sim time  ---

    if timestep_val is not None and run_val is not None:
        total_time = timestep_val * run_val
        if abs(total_time - expected_total_time1) > total_time_tol1:
            issues.append("sim time1 inaccurate")

    else:
        issues.append("total_time_incomplete") 
    
#     print(issues)
    
    return True if not issues else ", ".join(sorted(set(issues)))



In [16]:
def add_accuracy_from_ast_prompt3(df, **kwargs):
    df = df.copy()
    df["accurate"] = df.apply(lambda r: accuracy_from_ast_row_prompt3(r, **kwargs), axis=1)
    return df

def highlight_unparsed(row):
    if row["parsed"] != True:
        return ["background-color: #ffe6e6"] * len(row)  # light red
    return [""] * len(row)

accuracy_add3_df = add_accuracy_from_ast_prompt3(accuracy_df[accuracy_df['prompt'] == 'prompt3'])

a = accuracy_add3_df.style.apply(highlight_unparsed, axis=1)
display(a)

['velocity', 'all', 'create', '300', '5812775', 'dist', 'gaussian']
['velocity', 'all', 'set', '0', '0', '20', 'units', 'box']
20.0


,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,"['ERROR: Incorrect args for pair coefficients (src/src/MANYBODY/pair_eam.cpp:374)', 'Last command: pair_coeff * * potentials/prompt3.potential Nb Nb']",[Errno 2] No such file or directory: '/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial',"box_region_missing, fixed bounds, gap_size, pair_style inaccurate, projectile_region_size, run num inaccurate, sim time1 inaccurate, target_region_size, v_set_direction"


In [17]:
# print(prompt3)
# accuracy_add_df
# accuracy_add2_df
# accuracy_add3_df

df_all_accuracy_add = pd.concat([accuracy_add_df, accuracy_add2_df, accuracy_add3_df], axis=0, ignore_index=True)
# display(df_all_accuracy_add)

a = df_all_accuracy_add.style.apply(highlight_unparsed, axis=1)
display(a)

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True,n/a,True
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,"['ERROR: Incorrect args for pair coefficients (src/src/MANYBODY/pair_eam.cpp:374)', 'Last command: pair_coeff * * potentials/prompt2.potential Ni']",[Errno 2] No such file or directory: '/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial',"pair_style inaccurate, size"
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,"['ERROR: Incorrect args for pair coefficients (src/src/MANYBODY/pair_eam.cpp:374)', 'Last command: pair_coeff * * potentials/prompt3.potential Nb Nb']",[Errno 2] No such file or directory: '/apps/share64/debian10/lammps/lammps-02Aug23/bin/lmp_serial',"box_region_missing, fixed bounds, gap_size, pair_style inaccurate, projectile_region_size, run num inaccurate, sim time1 inaccurate, target_region_size, v_set_direction"


Checklist:
- Niobium
- EAM potential 
- Lattice Parameter ~ 3.3 Å
- Lattice Type: BCC

- NVT @ 300 for 100ps 
- NVE for some time
- Relative velocity of 2km/s and in [001]

- Run count $\mathring{\text{A}}$ 

- Free Boundary (p p s) or enough space

Size considerations:
- Projectile: 20 x 3.3Å = 60Å x 60Å x 60Å 
- Gap: 15Å 
- Target: 40 x 3.3Å = 120Å x 60Å x 60Å 
- Total: 195Å x 60Å x 60Å 

In [18]:
prompt3=(r"""Method Description: 

We simulate spall failure on Nb single crystals using high-velocity impact simulations using molecular dynamics (MD) with the LAMMPS code. We simulate the impact of a projectile on a target with a relative velocity of 2 km/s. The projectile is obtained by replicating the Nb BCC unit cell 20 times along the [100], [010], and [001]; the target is longer along the shock direction and is obtained by replicating the BCC unit cell 20 times along [100] and [010] and 40 times along [001]. We apply periodic boundary conditions along the directions normal to the impact direction, [001], and free boundaries along [001]. A gap of 1.5 nm initially separates the target and projectile. The system is equilibrated at 300 K for 100 ps using isothermal, isochoric MD. An impact velocity of 2 km/s is added to the thermal velocities to all the atoms in the projectile along [001] in the direction of the target. Adiabatic MD is used to simulate the impact and subsequent expansion. All atomic interactions are described using an EAM potential developed by Fellinger et al. [1] and downloaded from openKIM [2]. [1] Fellinger MR, Park H, Wilkins JW. Force-matched embedded-atom method potential for niobium. Physical Review B. 2010Apr;81(14):144119. doi:10.1103/PhysRevB.81.144119 [2] https://doi.org/10.25950/befb2eea.""")

print(prompt3)

Method Description: 

We simulate spall failure on Nb single crystals using high-velocity impact simulations using molecular dynamics (MD) with the LAMMPS code. We simulate the impact of a projectile on a target with a relative velocity of 2 km/s. The projectile is obtained by replicating the Nb BCC unit cell 20 times along the [100], [010], and [001]; the target is longer along the shock direction and is obtained by replicating the BCC unit cell 20 times along [100] and [010] and 40 times along [001]. We apply periodic boundary conditions along the directions normal to the impact direction, [001], and free boundaries along [001]. A gap of 1.5 nm initially separates the target and projectile. The system is equilibrated at 300 K for 100 ps using isothermal, isochoric MD. An impact velocity of 2 km/s is added to the thermal velocities to all the atoms in the projectile along [001] in the direction of the target. Adiabatic MD is used to simulate the impact and subsequent expansion. All at

In [19]:
display(df_all_accuracy_add[(df_all_accuracy_add['prompt']=='prompt1') & (df_all_accuracy_add['model']=='gpt-4o') ])


,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True,n/a,True


In [20]:
print(prompt_model_map1)

{'prompt1': ['gpt-4o'], 'prompt2': ['gpt-4o'], 'prompt3': ['gpt-4o']}


In [21]:
display(df_all_accuracy_add.head())

,prompt,model,trial,sanitized,parsed,ast_path,run,pair_run,accurate
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl,True,n/a,True
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...,[Errno 2] No such file or directory: '/apps/sh...,"pair_style inaccurate, size"
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl,['ERROR: Incorrect args for pair coefficients ...,[Errno 2] No such file or directory: '/apps/sh...,"box_region_missing, fixed bounds, gap_size, pa..."


In [22]:
# accuracy_df.to_pickle('150_scripts_5_models_sani.pkl')
df_all_accuracy_add.to_pickle('accuracy_df_trees.pkl')